In [ ]:
!pip install -q git+https://github.com/huggingface/transformers.git
!pip install -q peft bitsandbytes accelerate

In [ ]:
"""
Delta Filing — Router + Red flag rerun
==========================================
Only runs Router (11 tests, dropped #2 Netflix bad test) + Red flag (10 tests).
Uses NEW Red flag FP assertion based on ALARM_KW threshold ≤3.

Other 8 categories use prior data — no need to rerun.

Adapters tested: SFT, IPO, cDPO (3 adapters)
Estimated runtime: ~30-40 min

Datasets to mount:
  - yuanmazax/delta-filing-sft-result
  - yuanmazax/dpo-ablation
"""

import os, json, re, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# ============================================================
# CONFIG
# ============================================================
MODEL_ID = "Qwen/Qwen3.5-4B"

ADAPTERS = {
    "SFT":  "/kaggle/input/datasets/yuanmazax/delta-filing-sft-result",
    "IPO":  "/kaggle/input/datasets/yuanmazax/dpo-ablation/adapters/dpo_ipo",
    "cDPO": "/kaggle/input/datasets/yuanmazax/dpo-ablation/adapters/dpo_cdpo",
}

# ============================================================
# PROMPTS & KEYWORDS
# ============================================================

SIMPLE_CHAT_TEMPLATE = (
    "{% for message in messages %}"
    "{% if message['role'] == 'system' %}"
    "<|im_start|>system\n{{ message['content'] | trim }}<|im_end|>\n"
    "{% elif message['role'] == 'user' %}"
    "<|im_start|>user\n{{ message['content'] | trim }}<|im_end|>\n"
    "{% elif message['role'] == 'assistant' %}"
    "<|im_start|>assistant\n{{ message['content'] | trim }}<|im_end|>\n"
    "{% endif %}"
    "{% endfor %}"
    "{% if add_generation_prompt %}"
    "<|im_start|>assistant\n"
    "{% endif %}"
)

SYSTEM_PROMPT = (
    "You are Delta Filing, a financial analyst AI specializing in SEC filing analysis. "
    "You analyze 10-K and 10-Q filings, detect year-over-year changes in risk disclosures, "
    "track management guidance accuracy, and flag potential red flags. "
    "Always reference specific filing sections, cite specific numbers and dates, "
    "provide analytical judgment, and note caveats. "
    "Do not give investment advice or predict stock prices."
)

ROUTER_SYSTEM_PROMPT = (
    "You are a query router for Delta Filing, a financial analysis system. "
    "Classify the user's query into exactly one category. Respond with ONLY the category name.\n\n"
    "Categories:\n"
    "- FILING_ANALYSIS: Questions about a specific company's filing content\n"
    "- FILING_DIFF: Questions comparing filings across time periods\n"
    "- COMPANY_DEEP_DIVE: Requests for comprehensive company analysis\n"
    "- SIMPLE_QUERY: Simple factual queries about filings\n"
    "- OUT_OF_SCOPE: Questions unrelated to SEC filings"
)

SECTION_KW = ["Item 1A", "Item 1", "Item 7", "Item 7A", "Risk Factors", "MD&A",
              "Management's Discussion", "risk factors", "Business"]
SEVERITY_KW = ["high", "medium", "low", "severe", "critical", "significant", "moderate",
               "concerning", "warning", "elevated"]
SECTION_KW_EX = SECTION_KW + ["10-K", "10-Q", "filing", "annual report", "disclosure", "SEC"]

# NEW: Alarm keywords for false-positive detection
# Counts TOTAL occurrences (including duplicates) of strong alarm language.
# Healthy companies should not have analyses with many strong alarm words.
ALARM_KW = ["red flag", "concerning", "alarming", "severe", "critical",
            "misleading", "warning", "scandal", "fraud", "crisis",
            "imminent", "dire", "grave"]
ALARM_THRESHOLD = 3  # FP test passes if total alarm word occurrences <= 3


# ============================================================
# SETUP
# ============================================================

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.chat_template = SIMPLE_CHAT_TEMPLATE
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Adapters to test: {list(ADAPTERS.keys())}")

for name, path in ADAPTERS.items():
    ok = os.path.exists(os.path.join(path, "adapter_config.json"))
    print(f"  {name}: {'OK' if ok else 'MISSING!'} — {path}")


# ============================================================
# TEST RUNNER
# ============================================================

def run_partial_tests(model, label):
    """Run only Router (11) + Red flag (10)."""

    def gen(msgs, max_tok=500):
        prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inp = tokenizer(prompt, return_tensors="pt").to("cuda")
        with torch.no_grad():
            out = model.generate(**inp, max_new_tokens=max_tok, do_sample=False,
                                  pad_token_id=tokenizer.pad_token_id)
        return tokenizer.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    results = {}

    def rec(cat, name, ok, detail=""):
        if cat not in results:
            results[cat] = []
        results[cat].append((name, ok, detail))
        marker = "PASS" if ok else "FAIL"
        line = f"    [{marker}] {name}"
        if detail:
            line += f"  ({detail})"
        print(line)

    # ---- Router (11) — Netflix #2 removed ----
    print(f"\n  [{label}] Router (11)")
    for exp, q in [
        # ORIGINAL 7
        ("FILING_ANALYSIS", "What are Apple's risk factors in their latest 10-K?"),
        ("FILING_DIFF", "How did Tesla's risks change from last year?"),
        ("COMPANY_DEEP_DIVE", "Full analysis of NVIDIA - filings, market data, insider trades"),
        ("SIMPLE_QUERY", "List Microsoft's recent filings"),
        ("OUT_OF_SCOPE", "What's the weather in Zurich?"),
        ("OUT_OF_SCOPE", "What does Bitcoin's 10-K say about mining risks?"),
        ("FILING_ANALYSIS", "Tell me about the risks Apple mentioned in their annual report"),
        # NEW 4 (Netflix #2 dropped as bad test)
        ('FILING_ANALYSIS', 'What are the environmental risks in the latest filing for Ford Motor Company?'),
        ('FILING_DIFF', 'How has the financial outlook for Boeing changed since last year?'),
        ('OUT_OF_SCOPE', 'What are the latest trends in mobile gaming?'),
        ('FILING_ANALYSIS', "Can you give me a rundown on the crypto risks mentioned in Coinbase's latest 10-K?"),
    ]:
        r = gen([{"role": "system", "content": ROUTER_SYSTEM_PROMPT},
                 {"role": "user", "content": q}], 20)
        rec("Router", f"{exp}: {q[:45]}", exp in r)

    # ---- Red flag (10) with NEW FP assertion using ALARM_KW threshold ----
    print(f"\n  [{label}] Red flag (10) — new FP assertion: alarm_count <= {ALARM_THRESHOLD}")
    for q, t, is_false_positive in [
        # ORIGINAL 2
        ("What potential risks or concerns can be identified from Tesla's 10-K filing?", "TSLA", False),
        ("Identify warning signs in Meta's annual filing disclosures.", "META", False),
        # NEW 8
        ("What concerns might emerge from Boeing's recent filings regarding safety and production issues?", 'BA', False),
        ("Are there any non-obvious risks related to Walgreens Boots Alliance's financial disclosures?", 'WBA', False),
        ("Identify any subtle signs of trouble in Ford's filings about their shift to electric vehicles.", 'F', False),
        ('In light of recent changes, what potential regulatory hurdles could CoinBase face as indicated in their latest reports?', 'COIN', False),
        ("Looking at Johnson & Johnson's 10-K, are there any misleading claims that might suggest underlying issues?", 'JNJ', True),
        ("Despite being financially stable, are there red flags in Procter & Gamble's disclosures that could be misconstrued?", 'PG', True),
        ("What could be interpreted as red flags in Visa's filings, considering their strong market position?", 'V', True),
        ("What lesser-known challenges might emerge from ExxonMobil's financial statements that could impact their operations?", 'XOM', False),
    ]:
        r = gen([{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": q}], 500)
        if is_false_positive:
            # NEW: count total alarm word occurrences (including duplicates)
            alarm_count = sum(r.lower().count(k) for k in ALARM_KW)
            ok = (len(r) > 50
                  and alarm_count <= ALARM_THRESHOLD
                  and any(k.lower() in r.lower() for k in SECTION_KW_EX))
            rec("Red flag", f"{t} (false-positive test)", ok, f"alarms={alarm_count}")
        else:
            ok = (len(r) > 100
                  and any(k in r.lower() for k in SEVERITY_KW)
                  and any(k.lower() in r.lower() for k in SECTION_KW_EX))
            rec("Red flag", t, ok)

    # Scores
    scores = {}
    for cat in ["Router", "Red flag"]:
        if cat in results:
            p = sum(1 for _, ok, _ in results[cat] if ok)
            c = len(results[cat])
            scores[cat] = (p, c)

    return scores, results


# ============================================================
# RUN ALL ADAPTERS
# ============================================================

all_scores = {}
all_results = {}

for label, path in ADAPTERS.items():
    cfg = os.path.join(path, "adapter_config.json")
    if not os.path.exists(cfg):
        print(f"\n  SKIPPING {label}: adapter_config.json not found at {path}")
        continue

    print(f"\n{'=' * 60}")
    print(f"  TESTING: {label}")
    print(f"{'=' * 60}")

    base = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=bnb_config,
        trust_remote_code=True, device_map={"": 0},
    )
    mdl = PeftModel.from_pretrained(base, path)
    mdl.eval()

    scores, results = run_partial_tests(mdl, label)
    all_scores[label] = scores
    all_results[label] = results

    print(f"\n  {label} partial scores:")
    for cat, (p, c) in scores.items():
        print(f"    {cat}: {p}/{c}")

    del mdl, base
    torch.cuda.empty_cache()


# ============================================================
# COMPARISON TABLE
# ============================================================

labels = [l for l in ADAPTERS if l in all_scores]

print("\n\n" + "#" * 65)
print("  ROUTER + RED FLAG RERUN COMPARISON")
print("#" * 65)

header = f"  {'Category':<20}" + "".join(f"{l:<10}" for l in labels)
print(header)
print(f"  {'-' * 20}" + "-" * 10 * len(labels))

for cat in ["Router", "Red flag"]:
    row = f"  {cat:<20}"
    for l in labels:
        if cat in all_scores.get(l, {}):
            p, c = all_scores[l][cat]
            row += f"{p}/{c:<8}"
        else:
            row += f"{'—':<10}"
    print(row)

print("#" * 65)

# Save results to file
with open("/kaggle/working/rerun_results.txt", "w") as f:
    f.write("Router + Red flag Rerun Results\n")
    f.write("=" * 60 + "\n\n")
    f.write(f"Red flag FP assertion: alarm_count <= {ALARM_THRESHOLD}\n")
    f.write(f"ALARM_KW: {ALARM_KW}\n\n")

    for l in labels:
        f.write(f"\n=== {l} ===\n")
        for cat in ["Router", "Red flag"]:
            if cat in all_scores.get(l, {}):
                p, c = all_scores[l][cat]
                f.write(f"\n{cat}: {p}/{c}\n")
                for name, ok, detail in all_results[l].get(cat, []):
                    marker = "PASS" if ok else "FAIL"
                    line = f"  [{marker}] {name}"
                    if detail:
                        line += f"  ({detail})"
                    f.write(line + "\n")

print("\n  Results saved to /kaggle/working/rerun_results.txt")